# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [1]:
import os
from google import genai

# Inicializa el cliente. 
API_KEY_GEMINI = "AQ.Ab8RN6LwKvr_SMk5ejbJXfiJpJ-53f5B-t9iuHWJplMTYqLu6w"

client = genai.Client(api_key=API_KEY_GEMINI)

# Probamos una respuesta básica
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Hola Gemini, confirma que estás conectado correctamente.',
)

print(response.text)

¡Hola! Sí, estoy aquí y funcionando correctamente. ¿En qué puedo ayudarte hoy?


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

# Cargamos el subset de entrenamiento 
newsgroups_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

# Creamos el DataFrame
df = pd.DataFrame({
    'documento': newsgroups_data.data,
    'categoria': [newsgroups_data.target_names[t] for t in newsgroups_data.target]
})

# Limpiamos un poco eliminando filas vacías o con solo espacios en blanco
df = df[df['documento'].str.strip() != '']
df = df.reset_index(drop=True)

# Para no agotar la cuota de la API , limitaremos el corpus a los primeros 100 documentos
df = df.head(100)

print(f"Corpus cargado con éxito. Total de documentos seleccionados: {len(df)}")
df.head()

Corpus cargado con éxito. Total de documentos seleccionados: 100


,documento,categoria
0,I was wondering if anyone out there could enli...,rec.autos
1,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware
2,"well folks, my mac plus finally gave up the gh...",comp.sys.mac.hardware
3,\nDo you have Weitek's address/phone number? ...,comp.graphics
4,"From article <C5owCB.n3p@world.std.com>, by to...",sci.space


### 2.2 Transformo a embeddings

En esta celda vamos a iterar sobre los documentos del DataFrame, llamar a la API de Gemini para generar sus representaciones vectoriales (embeddings) usando el modelo especializado text-embedding-004, y guardarlas en una nueva lista/columna.

In [13]:
import time
import numpy as np

embeddings = []

print("Generando embeddings para los documentos...")
for i, texto in enumerate(df['documento']):
    try:
        # Usamos el modelo exacto que vimos en tu lista
        response = client.models.embed_content(
            model="models/gemini-embedding-2", 
            contents=texto
        )
        
        embedding_vector = response.embeddings[0].values
        embeddings.append(embedding_vector)
        
        # Delay corto de cortesía para la API gratuita
        time.sleep(0.5) 
        
    except Exception as e:
        print(f"Error en el documento índice {i}: {e}")
        break

# Guardamos en el DataFrame si todo coincide
if len(embeddings) == len(df):
    df['embedding'] = embeddings
    print("¡Embeddings generados y guardados con éxito en el DataFrame usando models/gemini-embedding-2!")
else:
    print(f"Ocurrió un problema. Se generaron {len(embeddings)} embeddings de {len(df)} documentos.")

Generando embeddings para los documentos...
¡Embeddings generados y guardados con éxito en el DataFrame usando models/gemini-embedding-2!


### 2.3 Creo una query y hago la búsqueda
Aquí definiremos una consulta de búsqueda (query) en texto plano, la transformaremos en su respectivo embedding usando el mismo modelo y calcularemos la similitu

In [14]:
import numpy as np

# Definimos la consulta que queremos hacer sobre el corpus
query = "Space exploration and NASA missions"

# Usamos exactamente el mismo modelo
response_query = client.models.embed_content(
    model="models/gemini-embedding-2",
    contents=query
)
query_embedding = np.array(response_query.embeddings[0].values)

print(f"Query definida: '{query}' y transformada a embedding con éxito usando models/gemini-embedding-2.")

Query definida: 'Space exploration and NASA missions' y transformada a embedding con éxito usando models/gemini-embedding-2.


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculamos la similitud de coseno entre el embedding de la query y todos los del DataFrame
similitudes = cosine_similarity([query_embedding], np.vstack(df['embedding'].values))[0]

# Agregamos los scores de similitud al DataFrame
df['similitud'] = similitudes

# Ordenamos de mayor a menor similitud y tomamos los 5 mejores
top_5_resultados = df.sort_values(by='similitud', ascending=False).head(5)

Obtengo los 5 documentos más similares a mi query

In [16]:
print(f"--- Top 5 documentos más similares para la búsqueda: '{query}' ---\n")

for idx, fila in top_5_resultados.iterrows():
    print(f"Ranking {top_5_resultados.index.get_loc(idx) + 1}")
    print(f"Categoría original: {fila['categoria']}")
    print(f"Score de Similitud: {fila['similitud']:.4f}")
    # Mostramos los primeros 250 caracteres del documento para no inundar la pantalla
    print(f"Contenido: {fila['documento'][:250].strip()}...")
    print("-" * 60)

--- Top 5 documentos más similares para la búsqueda: 'Space exploration and NASA missions' ---

Ranking 1
Categoría original: comp.sys.mac.hardware
Score de Similitud: 0.6227
Contenido: --...
------------------------------------------------------------
Ranking 2
Categoría original: sci.space
Score de Similitud: 0.6190
Contenido: From article <C5owCB.n3p@world.std.com>, by tombaker@world.std.com (Tom A Baker):


My understanding is that the 'expected errors' are basically
known bugs in the warning system software - things are checked
that don't have the right values in yet be...
------------------------------------------------------------
Ranking 3
Categoría original: sci.space
Score de Similitud: 0.5902
Contenido: Pat sez;

Yeah, but a windscreen cut down most of it.  Canopies ended it completely.

Of course, the environment in space continues to suck :-)

-Tommy Mac
-------------------------------------------------------------------------
Tom McWilliams 517-3...
----------------------